# 01 - Data preparation (Houston, 2024)

Build the POI-level tri-modal training table used by BEACON.

| Modality | Source | Dimensions |
|---|---|---:|
| AlphaEarth (local and context views) | Google Satellite Embedding V1, averaged over 9 x 9 and 19 x 19 pixel neighborhoods around each POI | 64 + 64 |
| POI semantics | SafeGraph POI attributes, templated descriptions, and `text-embedding-3-large` | 384 |
| Visitation behavior | SafeGraph hourly visits summarized as 168 hour-of-week values, 12 monthly values, and four scalar descriptors | 184 |

**Workflow:** export weekly SafeGraph records, extract POI coordinates, embed POI descriptions, build visitation profiles, sample AlphaEarth around POIs, merge the three views, and export the regional AlphaEarth raster used by notebooks 02 and 03.

**Primary output:** `contrastive_learning/houston_poi_ae_text_mobility_cov1_train.parquet`


## 0. Configuration


In [ ]:
from pathlib import Path
import os
import json
import time

import numpy as np
import pandas as pd
import duckdb

PROJECT_DIR = Path(os.environ.get("BEACON_PROJECT_DIR", Path.cwd())).expanduser().resolve()

# ---- Inputs ----
CBG_LIST_CSV = PROJECT_DIR / "Houston_cbg_list.csv"                    # Houston MSA census block groups
TRACT_GPKG   = PROJECT_DIR / "houston_msa" / "houston_msa_2024_tract.gpkg"

# ---- Intermediate outputs ----
RAW_MOBILITY_DIR = PROJECT_DIR / "houston_mobility_2024"               # Weekly SafeGraph Parquet files
POI_COORDS_CSV   = PROJECT_DIR / "houston_poi_coords_2024.csv"
POI_TEXT_PARQUET = PROJECT_DIR / "houston_poi_text_2024.parquet"
TEXT_EMB_PARQUET = PROJECT_DIR / "houston_poi_text_embeddings_384.parquet"
MOBILITY_DIR     = PROJECT_DIR / "houston_mobility_2024_embedding" / "mobility_profile_2024_global"
MOBILITY_PROFILE = MOBILITY_DIR / "houston_poi_mobility_profile_2024_186cols.parquet"
MOBILITY_COV1    = MOBILITY_DIR / "houston_poi_mobility_cov1_186cols.parquet"
AE_POI_CSV_DIR   = PROJECT_DIR / "houston_alphaearth_2024_tiles"       # POI-level AlphaEarth samples exported from Earth Engine
AE_RASTER_DIR    = PROJECT_DIR / "Houston_AE_2024"                     # 64-band GeoTIFF tiles exported from Earth Engine

# ---- Final output ----
CONTRASTIVE_DIR = PROJECT_DIR / "contrastive_learning"
TRAIN_TABLE     = CONTRASTIVE_DIR / "houston_poi_ae_text_mobility_cov1_train.parquet"

# ---- Earth Engine ----
GEE_PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT")
GEE_POI_ASSET = os.environ.get("BEACON_GEE_POI_ASSET")
AE_COLLECTION = "GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL"
YEAR_START, YEAR_END = "2024-01-01", "2025-01-01"

# ---- External services and cloud exports are disabled by default ----
RUN_MONGO_EXPORT      = False
RUN_TEXT_EMBEDDING    = False
RUN_GEE_POI_SAMPLING  = False
RUN_GEE_RASTER_EXPORT = False

for d in [RAW_MOBILITY_DIR, MOBILITY_DIR, AE_POI_CSV_DIR, AE_RASTER_DIR, CONTRASTIVE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RAW_GLOB = (RAW_MOBILITY_DIR / "*.parquet").as_posix()

## 1. Export Houston SafeGraph weekly records from MongoDB

Filter `safegraph.weekly2024` with the Houston MSA census block group list and write each 100,000 records to a Parquet file. Set `RUN_MONGO_EXPORT = True` to run this optional step.


In [ ]:
if RUN_MONGO_EXPORT:
    from pymongo import MongoClient

    collection = MongoClient(os.environ.get("MONGODB_URI", "mongodb://localhost:27017/"))["safegraph"]["weekly2024"]

    cbg = (
        pd.read_csv(CBG_LIST_CSV, dtype={"CensusBlockGroup": "string"})["CensusBlockGroup"]
        .str.extract(r"(\d+)", expand=False).str[:12].dropna()
    )
    cbg_str = sorted(set(cbg))
    cbg_int = [int(x) for x in cbg_str]          # poi_cbg may be stored as either a string or an integer

    query = {"$or": [{"poi_cbg": {"$in": cbg_str}}, {"poi_cbg": {"$in": cbg_int}}]}
    fields = [
        "placekey", "location_name", "naics_code", "top_category", "sub_category",
        "date_range_start", "date_range_end", "latitude", "longitude",
        "visits_by_each_hour", "raw_visit_counts", "raw_visitor_counts",
    ]
    projection = {"_id": 0, **{f: 1 for f in fields}}

    def flush(rows, part_id):
        path = RAW_MOBILITY_DIR / f"houston_part_{part_id:03d}.parquet"
        pd.DataFrame(rows).to_parquet(path, index=False)
        print(f"wrote {path.name}: {len(rows):,} rows")

    batch, part_id = [], 0
    with collection.find(query, projection=projection, batch_size=10_000, no_cursor_timeout=True) as cursor:
        for doc in cursor:
            batch.append(doc)
            if len(batch) >= 100_000:
                flush(batch, part_id)
                batch, part_id = [], part_id + 1
    if batch:
        flush(batch, part_id)

## 2. POI coordinates

Export one coordinate pair per POI. Upload the resulting CSV to Earth Engine as the asset specified by `BEACON_GEE_POI_ASSET` before running Section 5.


In [ ]:
poi_coords = duckdb.sql(f"""
    SELECT placekey,
           ANY_VALUE(latitude)  AS latitude,
           ANY_VALUE(longitude) AS longitude
    FROM '{RAW_GLOB}'
    GROUP BY placekey
""").df()

poi_coords.to_csv(POI_COORDS_CSV, index=False)
print(f"unique POIs: {len(poi_coords):,}")
print(poi_coords[["longitude", "latitude"]].agg(["min", "max"]))

## 3. POI descriptions and text embeddings

The description template is `A place of {sub_category}, a type of {top_category}, named {location_name}. The NAICS code is {naics_code}.` The code falls back to shorter templates when fields are missing.

Embeddings use `text-embedding-3-large` with `dimensions=384`. Read the API key and optional base URL from `OPENAI_API_KEY` and `OPENAI_BASE_URL`.


In [ ]:
poi_text = duckdb.sql(f"""
    WITH p AS (
        SELECT placekey,
               ANY_VALUE(location_name) AS location_name,
               ANY_VALUE(naics_code)    AS naics_code,
               ANY_VALUE(top_category)  AS top_category,
               ANY_VALUE(sub_category)  AS sub_category,
               ANY_VALUE(latitude)      AS latitude,
               ANY_VALUE(longitude)     AS longitude
        FROM '{RAW_GLOB}'
        GROUP BY placekey
    )
    SELECT *,
        CASE
            WHEN location_name IS NOT NULL AND sub_category IS NOT NULL
                 AND top_category IS NOT NULL AND naics_code IS NOT NULL
            THEN 'A place of ' || sub_category || ', a type of ' || top_category
                 || ', named ' || location_name || '. The NAICS code is '
                 || CAST(naics_code AS VARCHAR) || '.'
            WHEN location_name IS NOT NULL AND sub_category IS NOT NULL AND top_category IS NOT NULL
            THEN 'A place of ' || sub_category || ', a type of ' || top_category
                 || ', named ' || location_name || '.'
            WHEN sub_category IS NOT NULL AND top_category IS NOT NULL
            THEN 'A place of ' || sub_category || ', a type of ' || top_category || '.'
            WHEN location_name IS NOT NULL AND top_category IS NOT NULL
            THEN 'A place named ' || location_name || ', a type of ' || top_category || '.'
            WHEN location_name IS NOT NULL
            THEN 'A place named ' || location_name || '.'
            ELSE 'A point of interest.'
        END AS poi_text
    FROM p
""").df()

poi_text.to_parquet(POI_TEXT_PARQUET, index=False)
print(poi_text.shape)
poi_text[["placekey", "poi_text"]].head()

In [ ]:
if RUN_TEXT_EMBEDDING:
    from openai import OpenAI
    from tqdm.auto import tqdm

    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"], base_url=os.environ.get("OPENAI_BASE_URL"))
    texts = poi_text["poi_text"].fillna("A point of interest.").tolist()

    embeddings = []
    for i in tqdm(range(0, len(texts), 512)):
        response = client.embeddings.create(
            model="text-embedding-3-large", input=texts[i:i + 512], dimensions=384
        )
        embeddings.extend(item.embedding for item in response.data)

    emb = np.asarray(embeddings, dtype=np.float32)
    text_emb = pd.DataFrame(emb, columns=[f"text_emb_{i:03d}" for i in range(emb.shape[1])])
    text_emb.insert(0, "placekey", poi_text["placekey"].values)
    text_emb.insert(1, "poi_text", poi_text["poi_text"].values)
    text_emb.to_parquet(TEXT_EMB_PARQUET, index=False)
    print(text_emb.shape)

## 4. Visitation profiles (184 dimensions)

Clip each weekly 168-hour visit vector to January 1 through December 31, 2024 in `America/Chicago`, then aggregate the following features for each POI:

- `weekly_168_norm_*`: normalized hour-of-week distribution
- `monthly_12_norm_*`: normalized monthly distribution
- `visit_total_log`: log-transformed annual visits
- `entropy_168`: Shannon entropy of the hour-of-week distribution
- `weekday_weekend_gap`: cosine distance between weekday and weekend 24-hour profiles
- `monthly_cv`: coefficient of variation across monthly totals
- `coverage_ratio`: observed hours divided by the 8,784 hours in 2024

Only POIs with complete annual coverage are retained. Files are aggregated in batches and consolidated with DuckDB to limit memory use.


In [ ]:
TZ = "America/Chicago"
TARGET_START = pd.Timestamp("2024-01-01 00:00:00", tz=TZ)
TARGET_END = pd.Timestamp("2024-12-31 23:00:00", tz=TZ)
TARGET_HOURS = int((TARGET_END - TARGET_START) / pd.Timedelta(hours=1)) + 1   # 8784
FILES_PER_BATCH = 8

W_RAW = [f"weekly_168_raw_{i:03d}" for i in range(168)]
M_RAW = [f"monthly_12_raw_{m:02d}" for m in range(1, 13)]
TMP_DIR = MOBILITY_DIR / "tmp_raw_chunks"
TMP_DIR.mkdir(parents=True, exist_ok=True)


def parse_hourly(x):
    """Convert visits_by_each_hour to float32[168]; return None for invalid input."""
    if isinstance(x, str):
        try:
            x = json.loads(x)
        except ValueError:
            return None
    if x is None or len(x) != 168:
        return None
    return np.asarray(x, dtype=np.float32)


_week_cache = {}


def week_masks(week_start):
    """Return the 2024 mask, the 12 x 168 month matrix, and the number of in-window hours."""
    if week_start not in _week_cache:
        ts = pd.date_range(week_start, periods=168, freq="h")
        keep = (ts >= TARGET_START) & (ts <= TARGET_END)
        month_matrix = np.zeros((12, 168), dtype=np.float32)
        month_matrix[ts.month[keep] - 1, np.flatnonzero(keep)] = 1.0
        _week_cache[week_start] = (keep.astype(np.float32), month_matrix, int(keep.sum()))
    return _week_cache[week_start]


def aggregate_batch(files):
    paths = ", ".join(f"'{Path(p).as_posix()}'" for p in files)
    df = duckdb.sql(f"""
        SELECT placekey,
               CAST(date_range_start AS TIMESTAMPTZ) AS week_start,
               visits_by_each_hour
        FROM read_parquet([{paths}])
        WHERE placekey IS NOT NULL
    """).df()
    df["week_start"] = pd.to_datetime(df["week_start"], utc=True).dt.tz_convert(TZ)
    df["vec"] = df["visits_by_each_hour"].map(parse_hourly)
    df = df[df["vec"].notna()]

    parts = []
    for week_start, sub in df.groupby("week_start", sort=False):
        keep, month_matrix, n_hours = week_masks(week_start)
        V = np.vstack(sub["vec"].to_numpy())
        part = pd.DataFrame(np.hstack([V * keep, V @ month_matrix.T]), columns=W_RAW + M_RAW)
        part["observed_hours"] = n_hours
        part["placekey"] = sub["placekey"].to_numpy()
        parts.append(part)
    return pd.concat(parts, ignore_index=True).groupby("placekey", as_index=False).sum()


files = sorted(RAW_MOBILITY_DIR.glob("*.parquet"))
for b, start in enumerate(range(0, len(files), FILES_PER_BATCH), start=1):
    out = TMP_DIR / f"raw_batch_{b:04d}.parquet"
    if out.exists():
        continue
    aggregate_batch(files[start:start + FILES_PER_BATCH]).to_parquet(out, index=False)
    print(f"batch {b}: {out.name}")

In [ ]:
sum_cols = ", ".join(f"SUM({c}) AS {c}" for c in W_RAW + M_RAW + ["observed_hours"])
raw = duckdb.sql(f"""
    SELECT placekey, {sum_cols}
    FROM read_parquet('{(TMP_DIR / "*.parquet").as_posix()}')
    GROUP BY placekey
""").df()

W = raw[W_RAW].to_numpy(np.float64)
M = raw[M_RAW].to_numpy(np.float64)


def row_normalize(A):
    s = A.sum(axis=1, keepdims=True)
    return np.divide(A, s, out=np.zeros_like(A), where=s > 0)


w_norm, m_norm = row_normalize(W), row_normalize(M)

# Entropy of the hour-of-week distribution
entropy_168 = -np.sum(np.where(w_norm > 0, w_norm * np.log(np.where(w_norm > 0, w_norm, 1.0)), 0.0), axis=1)

# Cosine distance between weekday and weekend 24-hour profiles; weeks start on Monday
weekday = row_normalize(W[:, :120].reshape(-1, 5, 24).sum(axis=1))
weekend = row_normalize(W[:, 120:].reshape(-1, 2, 24).sum(axis=1))
denom = np.linalg.norm(weekday, axis=1) * np.linalg.norm(weekend, axis=1)
weekday_weekend_gap = np.where(denom > 0, 1.0 - (weekday * weekend).sum(axis=1) / np.where(denom > 0, denom, 1.0), 0.0)

# Coefficient of variation across monthly totals
m_mean = M.mean(axis=1)
monthly_cv = np.where(m_mean > 0, M.std(axis=1) / np.where(m_mean > 0, m_mean, 1.0), 0.0)

profile = pd.concat([
    pd.DataFrame({
        "placekey": raw["placekey"],
        "visit_total_log": np.log1p(W.sum(axis=1)),
        "coverage_ratio": raw["observed_hours"] / TARGET_HOURS,
        "entropy_168": entropy_168,
        "weekday_weekend_gap": weekday_weekend_gap,
        "monthly_cv": monthly_cv,
    }),
    pd.DataFrame(w_norm, columns=[f"weekly_168_norm_{i:03d}" for i in range(168)]),
    pd.DataFrame(m_norm, columns=[f"monthly_12_norm_{m:02d}" for m in range(1, 13)]),
], axis=1)

profile.to_parquet(MOBILITY_PROFILE, index=False)

cov1 = profile[profile["coverage_ratio"] == 1.0].reset_index(drop=True)
cov1.to_parquet(MOBILITY_COV1, index=False)

print(f"all POIs: {len(profile):,} | coverage_ratio == 1: {len(cov1):,}")
cov1[["visit_total_log", "entropy_168", "weekday_weekend_gap", "monthly_cv"]].describe().round(3)

## 5. Sample AlphaEarth at POIs with Earth Engine

Compute two neighborhood means from the 2024 AlphaEarth annual embedding and sample them at each POI:

- `small_A00` through `small_A63`: 9 x 9 pixels, approximately 90 m
- `context_A00` through `context_A63`: 19 x 19 pixels, approximately 190 m

The study area is divided into a 40 x 40 grid. Each tile includes a 300 m buffer to reduce edge effects. Exports are written to Google Drive. Download the completed CSV files to `AE_POI_CSV_DIR` before continuing.


In [ ]:
if RUN_GEE_POI_SAMPLING:
    if not GEE_PROJECT or not GEE_POI_ASSET:
        raise RuntimeError("Set GOOGLE_CLOUD_PROJECT and BEACON_GEE_POI_ASSET before running this section.")

    import ee

    ee.Authenticate(auth_mode="notebook")
    ee.Initialize(project=GEE_PROJECT)

    LON_MIN, LON_MAX = -96.600998, -94.359405   # POI coordinate extent from Section 2
    LAT_MIN, LAT_MAX = 28.874359, 30.601354
    N_LON = N_LAT = 40
    TILE_BUFFER_M = 300
    MAX_OUTSTANDING = 100
    TASK_PREFIX = "houston_ae_2024_tile"

    poi = ee.FeatureCollection(GEE_POI_ASSET).map(
        lambda f: f.set({"longitude": f.geometry().coordinates().get(0),
                         "latitude": f.geometry().coordinates().get(1)})
    )
    ae = ee.ImageCollection(AE_COLLECTION).filterDate(YEAR_START, YEAR_END).filterBounds(poi).mosaic()
    bands = ae.bandNames()

    def neighborhood_mean(image, radius, prefix):
        renamed = bands.map(lambda b: ee.String(prefix).cat(ee.String(b)))
        return image.reduceNeighborhood(
            reducer=ee.Reducer.mean(),
            kernel=ee.Kernel.square(radius=radius, units="pixels"),
            optimization="boxcar",
        ).rename(renamed)

    def our_tasks(states=None):
        tasks = [t for t in ee.batch.Task.list() if t.config.get("description", "").startswith(TASK_PREFIX)]
        if states:
            tasks = [t for t in tasks if t.status().get("state") in states]
        return tasks

    existing = {t.config["description"] for t in our_tasks()}

    for i in range(N_LON):
        for j in range(N_LAT):
            name = f"{TASK_PREFIX}_{i}_{j}"
            if name in existing:
                continue
            while len(our_tasks({"READY", "RUNNING"})) >= MAX_OUTSTANDING:
                time.sleep(60)

            lon0 = LON_MIN + (LON_MAX - LON_MIN) * i / N_LON
            lon1 = LON_MIN + (LON_MAX - LON_MIN) * (i + 1) / N_LON
            lat0 = LAT_MIN + (LAT_MAX - LAT_MIN) * j / N_LAT
            lat1 = LAT_MIN + (LAT_MAX - LAT_MIN) * (j + 1) / N_LAT
            tile = ee.Geometry.Rectangle([lon0, lat0, lon1, lat1])
            ae_tile = ae.clip(tile.buffer(TILE_BUFFER_M))

            sampled = (
                neighborhood_mean(ae_tile, 4, "small_")
                .addBands(neighborhood_mean(ae_tile, 9, "context_"))
                .sampleRegions(
                    collection=poi.filterBounds(tile),
                    properties=["placekey", "longitude", "latitude"],
                    scale=10, geometries=False, tileScale=16,
                )
            )
            ee.batch.Export.table.toDrive(
                collection=sampled, description=name, folder=AE_POI_CSV_DIR.name,
                fileNamePrefix=name, fileFormat="CSV",
            ).start()
            print("started", name)
            time.sleep(0.5)

## 6. Merge the tri-modal training table

Use the complete-coverage mobility table as the base and inner join text embeddings and AlphaEarth samples by `placekey`. POI metadata are retained for inspection.


In [ ]:
con = duckdb.connect()
con.execute("PRAGMA threads=8")

AE_GLOB = (AE_POI_CSV_DIR / "*.csv").as_posix()
ae_cols = ",\n        ".join(
    [f"ANY_VALUE({p}_A{i:02d}) AS {p}_A{i:02d}" for p in ("small", "context") for i in range(64)]
)

con.execute(f"""
CREATE OR REPLACE VIEW ae AS
SELECT placekey,
       ANY_VALUE(latitude)  AS ae_latitude,
       ANY_VALUE(longitude) AS ae_longitude,
       {ae_cols}
FROM read_csv_auto('{AE_GLOB}', header=True, union_by_name=True, files_to_sniff=-1)
GROUP BY placekey
""")

con.execute(f"""
COPY (
    SELECT m.* EXCLUDE (coverage_ratio),
           t.* EXCLUDE (placekey),
           a.* EXCLUDE (placekey),
           p.location_name, p.naics_code, p.top_category, p.sub_category,
           p.latitude  AS poi_latitude,
           p.longitude AS poi_longitude
    FROM '{MOBILITY_COV1.as_posix()}' m
    JOIN '{TEXT_EMB_PARQUET.as_posix()}' t USING (placekey)
    JOIN ae a USING (placekey)
    LEFT JOIN '{POI_TEXT_PARQUET.as_posix()}' p USING (placekey)
) TO '{TRAIN_TABLE.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

counts = con.execute(f"""
    SELECT
        (SELECT COUNT(*) FROM '{MOBILITY_COV1.as_posix()}') AS mobility_cov1,
        (SELECT COUNT(*) FROM '{TEXT_EMB_PARQUET.as_posix()}') AS text,
        (SELECT COUNT(*) FROM ae) AS alphaearth,
        (SELECT COUNT(*) FROM '{TRAIN_TABLE.as_posix()}') AS train_table
""").df()
con.close()
counts

## 7. Export the Houston AlphaEarth raster

Export the 2024 annual AlphaEarth embedding for the Houston MSA as 64-band, 10 m GeoTIFF tiles. Download the completed files from Google Drive to `AE_RASTER_DIR`.

This export follows the same tiling logic used in the related transfer experiments. Skip this section when `Houston_AE_2024/*.tif` is already available.


In [ ]:
if RUN_GEE_RASTER_EXPORT:
    if not GEE_PROJECT:
        raise RuntimeError("Set GOOGLE_CLOUD_PROJECT before running this section.")

    import ee
    import geopandas as gpd
    from shapely.geometry import box, mapping

    try:
        ee.Initialize(project=GEE_PROJECT)
    except Exception:
        ee.Authenticate(auth_mode="notebook")
        ee.Initialize(project=GEE_PROJECT)

    GRID_NX = GRID_NY = 8
    MAX_OUTSTANDING = 5
    TASK_PREFIX = "houston_alphaearth_2024_64band"

    msa = gpd.read_file(TRACT_GPKG).to_crs("EPSG:5070").dissolve()
    aoi_geom = msa.geometry.iloc[0].simplify(500, preserve_topology=True)
    aoi_wgs84 = gpd.GeoSeries([aoi_geom], crs="EPSG:5070").to_crs("EPSG:4326").iloc[0]
    aoi = ee.Geometry(mapping(aoi_wgs84))

    ae = (ee.ImageCollection(AE_COLLECTION).filterDate(YEAR_START, YEAR_END)
          .filterBounds(aoi).mosaic().toFloat().clip(aoi))

    def active_tasks():
        return sum(1 for t in ee.batch.Task.list()
                   if t.config.get("description", "").startswith(TASK_PREFIX)
                   and t.status().get("state") in {"READY", "RUNNING"})

    existing = {t.config.get("description", "") for t in ee.batch.Task.list()}
    minx, miny, maxx, maxy = aoi_wgs84.bounds
    for ix in range(GRID_NX):
        for iy in range(GRID_NY):
            x0, x1 = minx + (maxx - minx) * ix / GRID_NX, minx + (maxx - minx) * (ix + 1) / GRID_NX
            y0, y1 = miny + (maxy - miny) * iy / GRID_NY, miny + (maxy - miny) * (iy + 1) / GRID_NY
            name = f"{TASK_PREFIX}_{ix:02d}_{iy:02d}"
            if name in existing or not box(x0, y0, x1, y1).intersects(aoi_wgs84):
                continue
            while active_tasks() >= MAX_OUTSTANDING:
                time.sleep(30)
            ee.batch.Export.image.toDrive(
                image=ae, description=name, folder=AE_RASTER_DIR.name, fileNamePrefix=name,
                region=ee.Geometry.Rectangle([x0, y0, x1, y1], geodesic=False),
                scale=10, maxPixels=1e13, fileDimensions=3072, shardSize=256,
                skipEmptyTiles=True, fileFormat="GeoTIFF", formatOptions={"cloudOptimized": True},
            ).start()
            print("started", name)